In [148]:
import pandas as pd
import numpy as np
import subprocess
import os
from datetime import datetime, timedelta, time, date
import datetime
from datetime import datetime
import locale
import seaborn as sns
import matplotlib.pyplot as plt
import re
from collections import defaultdict

In [149]:
dia = "20260519"

Viaje desglosado

In [150]:
#Importar archivos desglosados

desglosado = pd.read_csv(f'Z:/01 base_datos/01 viajes_desglosados_FMS/{dia}_viajedesglosado.csv', encoding='latin')
    
desglosado.head()

,Fecha,Concesión,Concesionario de Operación Planificado,Concesionario de Operación Real,ServViaje,Servicio,Orden Viaje,Id Viaje,Viaje Línea,Id Línea,...,DistNoRealizada,KmEjecutado,DespInicial,Puntualidad Preliminar,IdValPuntualidad,Cumplimiento Preliminar,FraHorCump,IdValCumplimiento,ICK Full Preliminar,ICK Preliminar
0,19/05/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,(105) GMOVIL ENGATIVA,20260519-BC29D0002-1-12739,BC29D0002,1,2,1,10350,...,0,36341,Si,NaN,NaN,NaN,"1, 1",CDZ4,100.0,1.0
1,19/05/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,(105) GMOVIL ENGATIVA,20260519-BC29D0002-2-12738,BC29D0002,2,3,2,10350,...,0,36909,No,NaN,NaN,NaN,"1, 3",CDZ4,100.0,1.0
2,19/05/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,(105) GMOVIL ENGATIVA,20260519-BC29D0002-3-12739,BC29D0002,3,4,3,10350,...,0,36341,No,NaN,NaN,NaN,"1, 4",CDZ4,100.0,1.0
3,19/05/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,(105) GMOVIL ENGATIVA,20260519-BC29D0002-4-12738,BC29D0002,4,5,4,10350,...,0,36909,No,NaN,NaN,NaN,"1, 5",CDZ4,100.0,1.0
4,19/05/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,(105) GMOVIL ENGATIVA,20260519-BC29D0002-5-12739,BC29D0002,5,6,5,10350,...,0,36341,No,NaN,NaN,NaN,"1, 5",CDZ4,100.0,1.0


In [151]:
desglosado = desglosado.sort_values(
    ['Servicio', 'Orden Viaje ']
).reset_index(drop=True)


elim = desglosado[

    (
        desglosado['Eliminado']
        .astype(str)
        .str.upper()
        .str.strip()
        == 'ELIMINADO'
    )

    &

    (
        desglosado['Descripción Motivo Elim']
        .astype(str)
        .str.upper()
        .str.strip()
        == 'CONGESTION VEHICULAR'
    )

].copy()



elim['servicio_base'] = np.where(

    elim['ServBusRef'].notna(),

    elim['ServBusRef'],

    elim['Servicio']

)



resultado = []



for _, row in elim.iterrows():

    servicio_base = row['servicio_base']

    # Buscar todos los viajes relacionados
    temp = desglosado[

        (
            desglosado['Servicio'] == servicio_base
        )

        |

        (
            desglosado['ServBusRef'] == servicio_base
        )

    ].copy()

    # Si no encuentra datos continuar
    if temp.empty:
        continue

    # Marcar eliminado
    temp['es_eliminado'] = np.where(

        (
            temp['Orden Viaje '] == row['Orden Viaje ']
        )

        &

        (
            temp['Servicio'] == row['Servicio']
        ),

        1,

        0

    )

    # Datos referencia
    temp['servicio_base'] = servicio_base
    temp['servicio_eliminado'] = row['Servicio']

    resultado.append(temp)



if len(resultado) > 0:

    resultado_congestion = pd.concat(
        resultado,
        ignore_index=True
    )

    resultado_congestion = resultado_congestion.sort_values(
        [
            'servicio_base',
            'Orden Viaje '
        ]
    )

else:

    resultado_congestion = pd.DataFrame()

    print('No se encontraron registros')



resultado_congestion.head()

,Fecha,Concesión,Concesionario de Operación Planificado,Concesionario de Operación Real,ServViaje,Servicio,Orden Viaje,Id Viaje,Viaje Línea,Id Línea,...,Puntualidad Preliminar,IdValPuntualidad,Cumplimiento Preliminar,FraHorCump,IdValCumplimiento,ICK Full Preliminar,ICK Preliminar,es_eliminado,servicio_base,servicio_eliminado
0,19/05/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,(105) GMOVIL ENGATIVA,20260519-CE0630003-1-10533,CE0630003,1,2,1,10266,...,NaN,NaN,NaN,"1, 1",CDZ4,93.80,0.94,0,CE0630003,CE0630003
1,19/05/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,(105) GMOVIL ENGATIVA,20260519-CE0630003-2-10533,CE0630003,2,3,2,10266,...,NaN,NaN,NaN,"1, 2",CDZ4,93.62,0.94,0,CE0630003,CE0630003
2,19/05/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,(105) GMOVIL ENGATIVA,20260519-CE0630003-3-10533,CE0630003,3,4,3,10266,...,NaN,NaN,NaN,"1, 3",CDZ4,96.90,0.97,0,CE0630003,CE0630003
3,19/05/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,(105) GMOVIL ENGATIVA,20260519-CE0630003-4-10533,CE0630003,4,5,4,10266,...,NaN,NaN,NaN,"1, 3",CDZ4,100.00,0.93,0,CE0630003,CE0630003
4,19/05/2026,ENGATIVA ZN,(105) GMOVIL ENGATIVA,(105) GMOVIL ENGATIVA,20260519-CE0630003-5-10533,CE0630003,5,6,5,10266,...,NaN,NaN,NaN,"1, 4",CDZ4,90.19,0.91,0,CE0630003,CE0630003


Completar y comparar con datos de finalización del viaje anterior de la eliminación, o si es primer viaje

In [152]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
import pandas as pd

# Ruta archivo
archivo_salida = f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2025/Eliminaciones/delete_trips/{dia}_resultado_congestion.xlsx'

# Exportar dataframe
resultado_congestion.to_excel(
    archivo_salida,
    index=False
)

# Abrir excel
wb = load_workbook(archivo_salida)
ws = wb.active

# Color rojo
relleno_rojo = PatternFill(
    start_color='FF0000',
    end_color='FF0000',
    fill_type='solid'
)

# Buscar columna es_eliminado
col_es_eliminado = None

for cell in ws[1]:

    if cell.value == 'es_eliminado':

        col_es_eliminado = cell.column
        break

# Pintar filas
for row in range(2, ws.max_row + 1):

    valor = ws.cell(
        row=row,
        column=col_es_eliminado
    ).value

    if valor == 1:

        for col in range(1, ws.max_column + 1):

            ws.cell(
                row=row,
                column=col
            ).fill = relleno_rojo

# Guardar
wb.save(archivo_salida)

print('Archivo exportado correctamente')
print(archivo_salida)

Archivo exportado correctamente
C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2025/Eliminaciones/delete_trips/20260519_resultado_congestion.xlsx


In [153]:
resultado_congestion.to_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2025/Eliminaciones/delete_trips/{dia}_eliminaciones.csv', sep=';')

IPH

In [154]:
#Importar IPH

iph_zonal = pd.read_csv(f'Z:/01 base_datos/12 iph FMS/{dia}_iph_zonal.csv', encoding='latin')
iph_troncal = pd.read_csv(f'Z:/01 base_datos/34 iph alim_FMS/{dia}_iph_alim.csv', encoding='latin')

#Concatenado de dataframes IPH Completa

iph = pd.concat([iph_zonal,iph_troncal], ignore_index=True)

iph.head()

,Jornada,TipoDia,Concesionario de Operación,Instante,Servicio Bus,Evento,Id Línea,Tabla,Sublinea,Id Ruta,Id Nodo,Tipo Nodo,Viaje,Servicio Conductor entrante,Turno Entrante,Operador Entrante,Servicio Conductor Saliente,Tipo Vehiculo,Operador
0,GC260519T2,CN0010079501,GMOVIL ENGATIVA,3:20:00,CN31B0001,18,10339,1,NaN,NaN,142,Patio,1,CE101274,1.0,105.0,NaN,BUS (80),105
1,GC260519T2,CN0010079501,GMOVIL ENGATIVA,4:00:00,CN31B0001,4,10339,1,NaN,NaN,52372,Parada,1,NaN,NaN,NaN,NaN,BUS (80),105
2,GC260519T2,CN0010079501,GMOVIL ENGATIVA,4:00:00,CN31B0001,11,10339,1,2496.0,12756.0,52372,Parada,2,NaN,NaN,NaN,NaN,BUS (80),105
3,GC260519T2,CN0010079501,GMOVIL ENGATIVA,4:18:22,CN31B0001,0,10339,1,2496.0,12756.0,53222,Parada,2,NaN,NaN,NaN,NaN,BUS (80),105
4,GC260519T2,CN0010079501,GMOVIL ENGATIVA,4:26:24,CN31B0001,0,10339,1,2496.0,12756.0,52686,Parada,2,NaN,NaN,NaN,NaN,BUS (80),105


In [155]:
# Rellenar valores faltantes en Sublinea y Ruta basados en TipoDia, ServBus, y Coche
iph[['Sublinea', 'Id Ruta ']] = iph.groupby(['TipoDia', 'Servicio Bus', 'Tabla'])[['Sublinea', 'Id Ruta ']].transform(lambda x: x.ffill().bfill())

# Rellenar valores faltantes en ServicioCondEnt y ServicioCondSal basados en TipoDia, ServBus, Linea, Ruta, Coche y Viaje
iph[['Servicio Conductor entrante ', 'Servicio Conductor Saliente ']] = iph.groupby(['TipoDia', 'Servicio Bus', 'Id Línea', 'Id Ruta ', 'Tabla'])[['Servicio Conductor entrante ', 'Servicio Conductor Saliente ']].transform(lambda x: x.ffill().bfill())

iph.head()

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9404\4275713799.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  iph[['Servicio Conductor entrante ', 'Servicio Conductor Saliente ']] = iph.groupby(['TipoDia', 'Servicio Bus', 'Id Línea', 'Id Ruta ', 'Tabla'])[['Servicio Conductor entrante ', 'Servicio Conductor Saliente ']].transform(lambda x: x.ffill().bfill())


,Jornada,TipoDia,Concesionario de Operación,Instante,Servicio Bus,Evento,Id Línea,Tabla,Sublinea,Id Ruta,Id Nodo,Tipo Nodo,Viaje,Servicio Conductor entrante,Turno Entrante,Operador Entrante,Servicio Conductor Saliente,Tipo Vehiculo,Operador
0,GC260519T2,CN0010079501,GMOVIL ENGATIVA,3:20:00,CN31B0001,18,10339,1,2496.0,12756.0,142,Patio,1,CE101274,1.0,105.0,CE101274,BUS (80),105
1,GC260519T2,CN0010079501,GMOVIL ENGATIVA,4:00:00,CN31B0001,4,10339,1,2496.0,12756.0,52372,Parada,1,CE101274,NaN,NaN,CE101274,BUS (80),105
2,GC260519T2,CN0010079501,GMOVIL ENGATIVA,4:00:00,CN31B0001,11,10339,1,2496.0,12756.0,52372,Parada,2,CE101274,NaN,NaN,CE101274,BUS (80),105
3,GC260519T2,CN0010079501,GMOVIL ENGATIVA,4:18:22,CN31B0001,0,10339,1,2496.0,12756.0,53222,Parada,2,CE101274,NaN,NaN,CE101274,BUS (80),105
4,GC260519T2,CN0010079501,GMOVIL ENGATIVA,4:26:24,CN31B0001,0,10339,1,2496.0,12756.0,52686,Parada,2,CE101274,NaN,NaN,CE101274,BUS (80),105


In [156]:
# Filtrar el DataFrame para que solo incluya filas donde 'Evento' sea 3 o 11
iph= iph[(iph['Evento'] == 3) | (iph['Evento'] == 11)]

iph['Sublinea'] = iph['Sublinea'].astype(int)
iph['Ruta'] = iph['Id Ruta '].astype(int)

#Eliminar columnas que no se necesitan para el proceso
col_eliminar = ['Operador', 'Turno Entrante ', 'Operador Entrante ']

iph= iph.drop(columns=col_eliminar)

iph.head()

,Jornada,TipoDia,Concesionario de Operación,Instante,Servicio Bus,Evento,Id Línea,Tabla,Sublinea,Id Ruta,Id Nodo,Tipo Nodo,Viaje,Servicio Conductor entrante,Servicio Conductor Saliente,Tipo Vehiculo,Ruta
2,GC260519T2,CN0010079501,GMOVIL ENGATIVA,4:00:00,CN31B0001,11,10339,1,2496,12756.0,52372,Parada,2,CE101274,CE101274,BUS (80),12756
14,GC260519T2,CN0010079501,GMOVIL ENGATIVA,5:54:45,CN31B0001,3,10339,1,2496,12756.0,52372,Parada,3,CE101515,CE101274,BUS (80),12756
26,GC260519T2,CN0010079501,GMOVIL ENGATIVA,9:04:00,CN31B0001,3,10339,1,2496,12756.0,52372,Parada,4,CE101549,CE101515,BUS (80),12756
38,GC260519T2,CN0010079501,GMOVIL ENGATIVA,12:10:00,CN31B0001,3,10339,1,2496,12756.0,52372,Parada,5,CE101477,CE101549,BUS (80),12756
50,GC260519T2,CN0010079501,GMOVIL ENGATIVA,15:10:00,CN31B0001,3,10339,1,2496,12756.0,52372,Parada,6,CE101655,CE101477,BUS (80),12756


In [157]:
# Función para corregir tiempos con "24:00:00"
def fix_time(date_str):
    if '24:' in date_str:
        return date_str.replace('24:', '00:')
    elif '25:' in date_str:
        return date_str.replace('25:', '01:')
    elif '26:' in date_str:
        return date_str.replace('26:', '02:')
    elif '27:' in date_str:
        return date_str.replace('27:', '03:')
    elif '28:' in date_str:
        return date_str.replace('28:', '04:')
    elif '29:' in date_str:
        return date_str.replace('29:', '05:')
    else:
        return date_str

# Función para convertir tiempo a segundos
def time_to_seconds(time_obj):
    return time_obj.hour * 3600 + time_obj.minute * 60 + time_obj.second

# Convertir la columna de tiempo a cadenas
iph['Instante'] = iph['Instante'].astype(str)

# Aplicar la función para corregir los tiempos
iph['Instante'] = iph['Instante'].apply(fix_time)

# Convertir la columna de tiempo a datetime, usando errors='coerce' para manejar errores
iph['Instante'] = pd.to_datetime(iph['Instante'], format='%H:%M:%S', errors='coerce')

# Eliminar la fecha predeterminada para trabajar solo con la parte de tiempo
iph['Instante'] = iph['Instante'].dt.time

# Manejar NaT después de la conversión
iph['Instante'] = iph['Instante'].apply(lambda x: x if pd.notnull(x) else pd.Timestamp('00:00:00').time())

# Convertir la columna 'Instante' a segundos
iph['Instante_segundos'] = iph['Instante'].apply(time_to_seconds).astype(int)

iph.head()

,Jornada,TipoDia,Concesionario de Operación,Instante,Servicio Bus,Evento,Id Línea,Tabla,Sublinea,Id Ruta,Id Nodo,Tipo Nodo,Viaje,Servicio Conductor entrante,Servicio Conductor Saliente,Tipo Vehiculo,Ruta,Instante_segundos
2,GC260519T2,CN0010079501,GMOVIL ENGATIVA,04:00:00,CN31B0001,11,10339,1,2496,12756.0,52372,Parada,2,CE101274,CE101274,BUS (80),12756,14400
14,GC260519T2,CN0010079501,GMOVIL ENGATIVA,05:54:45,CN31B0001,3,10339,1,2496,12756.0,52372,Parada,3,CE101515,CE101274,BUS (80),12756,21285
26,GC260519T2,CN0010079501,GMOVIL ENGATIVA,09:04:00,CN31B0001,3,10339,1,2496,12756.0,52372,Parada,4,CE101549,CE101515,BUS (80),12756,32640
38,GC260519T2,CN0010079501,GMOVIL ENGATIVA,12:10:00,CN31B0001,3,10339,1,2496,12756.0,52372,Parada,5,CE101477,CE101549,BUS (80),12756,43800
50,GC260519T2,CN0010079501,GMOVIL ENGATIVA,15:10:00,CN31B0001,3,10339,1,2496,12756.0,52372,Parada,6,CE101655,CE101477,BUS (80),12756,54600


Tabla de eventos - Datos brutos

In [158]:
eventos_sae =pd.read_excel(f'Z:/01 base_datos/41 Informe Diario CCZ/FMS 2025/Eventos smart/{dia}_eventos smart.xlsx')

eventos_sae.head()

,APPLY_DATE,VEH_SERV_ID,SERV_TRIP_SEQ,VEH_REGISTR_NUM,EVENT_DATETIME,START_DATETIME,END_DATETIME,START_VEH_LAT,START_VEH_LON,END_VEH_LAT,END_VEH_LON,START_ROUTE_OFFSET_VALUE,END_ROUTE_OFFSET_VALUE,TRIP_END_TYPE_CD,CREAT_USER_ID,CREAT_DATETIME,UPD_USER_ID,UPD_DATETIME
0,20260519,AD0045123,1,10132,20260519052719,20260519052719,2.026052e+13,507235,588724,517432,597536,0,23778.0,NaN,SmartCore,20260519052742,SmartCore,2.026052e+13
1,20260519,AD0045123,2,10132,20260519063557,20260519063557,2.026052e+13,517528,597469,507054,588348,0,24797.0,NaN,SmartCore,20260519063600,SmartCore,2.026052e+13
2,20260519,AD0066049,1,7028,20260519095123,20260519095123,2.026052e+13,521962,605199,517528,597472,0,22012.0,NaN,SmartCore,20260519095132,SmartCore,2.026052e+13
3,20260519,AD0066050,1,7028,20260519194559,20260519194559,2.026052e+13,520031,604881,517525,597471,1967,22012.0,NaN,SmartCore,20260519194600,SmartCore,2.026052e+13
4,20260519,AD0067054,1,40407,20260519210330,20260519210330,2.026052e+13,509019,600319,505311,595454,0,24834.0,NaN,SmartCore,20260519210350,SmartCore,2.026052e+13


In [159]:
cols_fecha = [
    'EVENT_DATETIME'
]

for col in cols_fecha:

    # Convertir a numérico
    eventos_sae[col] = pd.to_numeric(
        eventos_sae[col],
        errors='coerce'
    )

    # Pasar a entero sin decimales
    eventos_sae[col] = (
        eventos_sae[col]
        .fillna(0)
        .astype('Int64')
        .astype(str)
    )

    # Convertir a datetime
    eventos_sae[col] = pd.to_datetime(
        eventos_sae[col],
        format='%Y%m%d%H%M%S',
        errors='coerce'
    )

eventos_sae[
    ['EVENT_DATETIME']
].head()

eventos_sae.head()

,APPLY_DATE,VEH_SERV_ID,SERV_TRIP_SEQ,VEH_REGISTR_NUM,EVENT_DATETIME,START_DATETIME,END_DATETIME,START_VEH_LAT,START_VEH_LON,END_VEH_LAT,END_VEH_LON,START_ROUTE_OFFSET_VALUE,END_ROUTE_OFFSET_VALUE,TRIP_END_TYPE_CD,CREAT_USER_ID,CREAT_DATETIME,UPD_USER_ID,UPD_DATETIME
0,20260519,AD0045123,1,10132,2026-05-19 05:27:19,20260519052719,2.026052e+13,507235,588724,517432,597536,0,23778.0,NaN,SmartCore,20260519052742,SmartCore,2.026052e+13
1,20260519,AD0045123,2,10132,2026-05-19 06:35:57,20260519063557,2.026052e+13,517528,597469,507054,588348,0,24797.0,NaN,SmartCore,20260519063600,SmartCore,2.026052e+13
2,20260519,AD0066049,1,7028,2026-05-19 09:51:23,20260519095123,2.026052e+13,521962,605199,517528,597472,0,22012.0,NaN,SmartCore,20260519095132,SmartCore,2.026052e+13
3,20260519,AD0066050,1,7028,2026-05-19 19:45:59,20260519194559,2.026052e+13,520031,604881,517525,597471,1967,22012.0,NaN,SmartCore,20260519194600,SmartCore,2.026052e+13
4,20260519,AD0067054,1,40407,2026-05-19 21:03:30,20260519210330,2.026052e+13,509019,600319,505311,595454,0,24834.0,NaN,SmartCore,20260519210350,SmartCore,2.026052e+13


In [160]:
cols_fecha = [
    'START_DATETIME',
    'END_DATETIME'
]

for col in cols_fecha:

    # Convertir a numérico
    eventos_sae[col] = pd.to_numeric(
        eventos_sae[col],
        errors='coerce'
    )

    # Pasar a entero sin decimales
    eventos_sae[col] = (
        eventos_sae[col]
        .fillna(0)
        .astype('Int64')
        .astype(str)
    )

    # Convertir a datetime
    eventos_sae[col] = pd.to_datetime(
        eventos_sae[col],
        format='%Y%m%d%H%M%S',
        errors='coerce'
    )

eventos_sae[
    ['START_DATETIME', 'END_DATETIME']
].head()

eventos_sae.head()

,APPLY_DATE,VEH_SERV_ID,SERV_TRIP_SEQ,VEH_REGISTR_NUM,EVENT_DATETIME,START_DATETIME,END_DATETIME,START_VEH_LAT,START_VEH_LON,END_VEH_LAT,END_VEH_LON,START_ROUTE_OFFSET_VALUE,END_ROUTE_OFFSET_VALUE,TRIP_END_TYPE_CD,CREAT_USER_ID,CREAT_DATETIME,UPD_USER_ID,UPD_DATETIME
0,20260519,AD0045123,1,10132,2026-05-19 05:27:19,2026-05-19 05:27:19,2026-05-19 06:33:41,507235,588724,517432,597536,0,23778.0,NaN,SmartCore,20260519052742,SmartCore,2.026052e+13
1,20260519,AD0045123,2,10132,2026-05-19 06:35:57,2026-05-19 06:35:57,2026-05-19 07:34:26,517528,597469,507054,588348,0,24797.0,NaN,SmartCore,20260519063600,SmartCore,2.026052e+13
2,20260519,AD0066049,1,7028,2026-05-19 09:51:23,2026-05-19 09:51:23,2026-05-19 10:47:05,521962,605199,517528,597472,0,22012.0,NaN,SmartCore,20260519095132,SmartCore,2.026052e+13
3,20260519,AD0066050,1,7028,2026-05-19 19:45:59,2026-05-19 19:45:59,2026-05-19 20:42:49,520031,604881,517525,597471,1967,22012.0,NaN,SmartCore,20260519194600,SmartCore,2.026052e+13
4,20260519,AD0067054,1,40407,2026-05-19 21:03:30,2026-05-19 21:03:30,2026-05-19 21:30:44,509019,600319,505311,595454,0,24834.0,NaN,SmartCore,20260519210350,SmartCore,2.026052e+13


In [161]:
eventos_sae['START_DATETIME'] = (
    eventos_sae['START_DATETIME']
    .dt.strftime('%H:%M:%S')
)

eventos_sae['END_DATETIME'] = (
    eventos_sae['END_DATETIME']
    .dt.strftime('%H:%M:%S')
)

eventos_sae['EVENT_DATETIME'] = (
    eventos_sae['EVENT_DATETIME']
    .dt.strftime('%H:%M:%S')
)

eventos_sae.head()

,APPLY_DATE,VEH_SERV_ID,SERV_TRIP_SEQ,VEH_REGISTR_NUM,EVENT_DATETIME,START_DATETIME,END_DATETIME,START_VEH_LAT,START_VEH_LON,END_VEH_LAT,END_VEH_LON,START_ROUTE_OFFSET_VALUE,END_ROUTE_OFFSET_VALUE,TRIP_END_TYPE_CD,CREAT_USER_ID,CREAT_DATETIME,UPD_USER_ID,UPD_DATETIME
0,20260519,AD0045123,1,10132,05:27:19,05:27:19,06:33:41,507235,588724,517432,597536,0,23778.0,NaN,SmartCore,20260519052742,SmartCore,2.026052e+13
1,20260519,AD0045123,2,10132,06:35:57,06:35:57,07:34:26,517528,597469,507054,588348,0,24797.0,NaN,SmartCore,20260519063600,SmartCore,2.026052e+13
2,20260519,AD0066049,1,7028,09:51:23,09:51:23,10:47:05,521962,605199,517528,597472,0,22012.0,NaN,SmartCore,20260519095132,SmartCore,2.026052e+13
3,20260519,AD0066050,1,7028,19:45:59,19:45:59,20:42:49,520031,604881,517525,597471,1967,22012.0,NaN,SmartCore,20260519194600,SmartCore,2.026052e+13
4,20260519,AD0067054,1,40407,21:03:30,21:03:30,21:30:44,509019,600319,505311,595454,0,24834.0,NaN,SmartCore,20260519210350,SmartCore,2.026052e+13


In [162]:
eventos_SAE = eventos_sae.copy()

In [163]:
#Filtrar lo que sea dferente a vacio en HInicio
eventos_SAE = eventos_SAE[
    (eventos_SAE['START_DATETIME'] != 0) &
    (eventos_SAE['START_DATETIME'] != '') &
    (eventos_SAE['START_DATETIME'].notna())
]
eventos_SAE.head()

,APPLY_DATE,VEH_SERV_ID,SERV_TRIP_SEQ,VEH_REGISTR_NUM,EVENT_DATETIME,START_DATETIME,END_DATETIME,START_VEH_LAT,START_VEH_LON,END_VEH_LAT,END_VEH_LON,START_ROUTE_OFFSET_VALUE,END_ROUTE_OFFSET_VALUE,TRIP_END_TYPE_CD,CREAT_USER_ID,CREAT_DATETIME,UPD_USER_ID,UPD_DATETIME
0,20260519,AD0045123,1,10132,05:27:19,05:27:19,06:33:41,507235,588724,517432,597536,0,23778.0,NaN,SmartCore,20260519052742,SmartCore,2.026052e+13
1,20260519,AD0045123,2,10132,06:35:57,06:35:57,07:34:26,517528,597469,507054,588348,0,24797.0,NaN,SmartCore,20260519063600,SmartCore,2.026052e+13
2,20260519,AD0066049,1,7028,09:51:23,09:51:23,10:47:05,521962,605199,517528,597472,0,22012.0,NaN,SmartCore,20260519095132,SmartCore,2.026052e+13
3,20260519,AD0066050,1,7028,19:45:59,19:45:59,20:42:49,520031,604881,517525,597471,1967,22012.0,NaN,SmartCore,20260519194600,SmartCore,2.026052e+13
4,20260519,AD0067054,1,40407,21:03:30,21:03:30,21:30:44,509019,600319,505311,595454,0,24834.0,NaN,SmartCore,20260519210350,SmartCore,2.026052e+13


In [164]:
#Filtrar lo que sea dferente a vacio en HFinal
eventos_SAE = eventos_SAE[
    (eventos_SAE['END_DATETIME'] != 0) &
    (eventos_SAE['END_DATETIME'] != '') &
    (eventos_SAE['END_DATETIME'].notna())
]
eventos_SAE.head()

,APPLY_DATE,VEH_SERV_ID,SERV_TRIP_SEQ,VEH_REGISTR_NUM,EVENT_DATETIME,START_DATETIME,END_DATETIME,START_VEH_LAT,START_VEH_LON,END_VEH_LAT,END_VEH_LON,START_ROUTE_OFFSET_VALUE,END_ROUTE_OFFSET_VALUE,TRIP_END_TYPE_CD,CREAT_USER_ID,CREAT_DATETIME,UPD_USER_ID,UPD_DATETIME
0,20260519,AD0045123,1,10132,05:27:19,05:27:19,06:33:41,507235,588724,517432,597536,0,23778.0,NaN,SmartCore,20260519052742,SmartCore,2.026052e+13
1,20260519,AD0045123,2,10132,06:35:57,06:35:57,07:34:26,517528,597469,507054,588348,0,24797.0,NaN,SmartCore,20260519063600,SmartCore,2.026052e+13
2,20260519,AD0066049,1,7028,09:51:23,09:51:23,10:47:05,521962,605199,517528,597472,0,22012.0,NaN,SmartCore,20260519095132,SmartCore,2.026052e+13
3,20260519,AD0066050,1,7028,19:45:59,19:45:59,20:42:49,520031,604881,517525,597471,1967,22012.0,NaN,SmartCore,20260519194600,SmartCore,2.026052e+13
4,20260519,AD0067054,1,40407,21:03:30,21:03:30,21:30:44,509019,600319,505311,595454,0,24834.0,NaN,SmartCore,20260519210350,SmartCore,2.026052e+13


In [165]:
#convertir columnas en entero
eventos_SAE['START_VEH_LAT'] = eventos_SAE['START_VEH_LAT'].astype(int)
eventos_SAE['START_VEH_LON']= eventos_SAE['START_VEH_LON'].astype(int)
eventos_SAE['END_VEH_LAT']= eventos_SAE['END_VEH_LAT'].astype(int)
eventos_SAE['END_VEH_LON']= eventos_SAE['END_VEH_LON'].astype(int)
eventos_SAE['START_ROUTE_OFFSET_VALUE']= eventos_SAE['START_ROUTE_OFFSET_VALUE'].astype(int)

eventos_SAE.head()

,APPLY_DATE,VEH_SERV_ID,SERV_TRIP_SEQ,VEH_REGISTR_NUM,EVENT_DATETIME,START_DATETIME,END_DATETIME,START_VEH_LAT,START_VEH_LON,END_VEH_LAT,END_VEH_LON,START_ROUTE_OFFSET_VALUE,END_ROUTE_OFFSET_VALUE,TRIP_END_TYPE_CD,CREAT_USER_ID,CREAT_DATETIME,UPD_USER_ID,UPD_DATETIME
0,20260519,AD0045123,1,10132,05:27:19,05:27:19,06:33:41,507235,588724,517432,597536,0,23778.0,NaN,SmartCore,20260519052742,SmartCore,2.026052e+13
1,20260519,AD0045123,2,10132,06:35:57,06:35:57,07:34:26,517528,597469,507054,588348,0,24797.0,NaN,SmartCore,20260519063600,SmartCore,2.026052e+13
2,20260519,AD0066049,1,7028,09:51:23,09:51:23,10:47:05,521962,605199,517528,597472,0,22012.0,NaN,SmartCore,20260519095132,SmartCore,2.026052e+13
3,20260519,AD0066050,1,7028,19:45:59,19:45:59,20:42:49,520031,604881,517525,597471,1967,22012.0,NaN,SmartCore,20260519194600,SmartCore,2.026052e+13
4,20260519,AD0067054,1,40407,21:03:30,21:03:30,21:30:44,509019,600319,505311,595454,0,24834.0,NaN,SmartCore,20260519210350,SmartCore,2.026052e+13


In [166]:
eventos_SAE['VEH_SERV_ID'] = eventos_SAE['VEH_SERV_ID'].replace('', 0).fillna(0)

eventos_SAE.head()

,APPLY_DATE,VEH_SERV_ID,SERV_TRIP_SEQ,VEH_REGISTR_NUM,EVENT_DATETIME,START_DATETIME,END_DATETIME,START_VEH_LAT,START_VEH_LON,END_VEH_LAT,END_VEH_LON,START_ROUTE_OFFSET_VALUE,END_ROUTE_OFFSET_VALUE,TRIP_END_TYPE_CD,CREAT_USER_ID,CREAT_DATETIME,UPD_USER_ID,UPD_DATETIME
0,20260519,AD0045123,1,10132,05:27:19,05:27:19,06:33:41,507235,588724,517432,597536,0,23778.0,NaN,SmartCore,20260519052742,SmartCore,2.026052e+13
1,20260519,AD0045123,2,10132,06:35:57,06:35:57,07:34:26,517528,597469,507054,588348,0,24797.0,NaN,SmartCore,20260519063600,SmartCore,2.026052e+13
2,20260519,AD0066049,1,7028,09:51:23,09:51:23,10:47:05,521962,605199,517528,597472,0,22012.0,NaN,SmartCore,20260519095132,SmartCore,2.026052e+13
3,20260519,AD0066050,1,7028,19:45:59,19:45:59,20:42:49,520031,604881,517525,597471,1967,22012.0,NaN,SmartCore,20260519194600,SmartCore,2.026052e+13
4,20260519,AD0067054,1,40407,21:03:30,21:03:30,21:30:44,509019,600319,505311,595454,0,24834.0,NaN,SmartCore,20260519210350,SmartCore,2.026052e+13


Acciones de regulación

In [167]:
#Tabla acciones regulación SAE

acciones_SAE = pd.read_csv(f'Z:/01 base_datos/09 acciones_regulacion_FMS/{dia}_accionesregulacion.csv',encoding='latin')

acciones_SAE.head()

,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,Nombre de Usuario,Parámetros,Id Motivo,Motivo,Reversada por
0,19/05/2026,0:12:49,10325,576,3.0,NaN,0.0,5,Eliminar Vehiculo,LJAIMES_112,Luis Eduardo Jaimes,"<= Elimination Vehicle =>\nIdLinea=10325,\nSer...",14,No se presenta conductor a realizar servicio,NaN
1,19/05/2026,0:13:16,10325,576,7.0,NaN,0.0,5,Eliminar Vehiculo,LJAIMES_112,Luis Eduardo Jaimes,"<= Elimination Vehicle =>\nIdLinea=10325,\nSer...",14,No se presenta conductor a realizar servicio,NaN
2,19/05/2026,0:13:33,10325,576,12.0,NaN,0.0,5,Eliminar Vehiculo,LJAIMES_112,Luis Eduardo Jaimes,"<= Elimination Vehicle =>\nIdLinea=10325,\nSer...",14,No se presenta conductor a realizar servicio,NaN
3,19/05/2026,0:28:48,10273,142,4.0,NaN,NaN,20,Cambiar Conductor,JBERNAL_105,JENNY BERNAL ALEXANDRA BERNAL,com.lgcns.fms.oprt.schedule.driver.model.Chang...,0,Motivo no definido.,NaN
4,19/05/2026,0:29:14,10273,142,15.0,NaN,NaN,20,Cambiar Conductor,JBERNAL_105,JENNY BERNAL ALEXANDRA BERNAL,com.lgcns.fms.oprt.schedule.driver.model.Chang...,0,Motivo no definido.,NaN


In [168]:
cols = [
    'Tabla',
    'Código Bus',
    'Número FMS Bus',
    'Reversada por'
]

# Reemplazar vacíos y NaN por 0
for col in cols:

    acciones_SAE[col] = (
        acciones_SAE[col]
        .replace([
            np.nan,
            'NaN',
            'nan',
            'NAN',
            '',
            'None',
            'NONE'
        ], 0)
        .fillna(0)
    )

# Convertir a entero solo estas columnas
cols_int = [
    'Tabla',
    'Número FMS Bus'
]

for col in cols_int:

    acciones_SAE[col] = pd.to_numeric(
        acciones_SAE[col],
        errors='coerce'
    ).fillna(0).astype(int)

acciones_SAE.head()

,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,Nombre de Usuario,Parámetros,Id Motivo,Motivo,Reversada por
0,19/05/2026,0:12:49,10325,576,3,0,0,5,Eliminar Vehiculo,LJAIMES_112,Luis Eduardo Jaimes,"<= Elimination Vehicle =>\nIdLinea=10325,\nSer...",14,No se presenta conductor a realizar servicio,0
1,19/05/2026,0:13:16,10325,576,7,0,0,5,Eliminar Vehiculo,LJAIMES_112,Luis Eduardo Jaimes,"<= Elimination Vehicle =>\nIdLinea=10325,\nSer...",14,No se presenta conductor a realizar servicio,0
2,19/05/2026,0:13:33,10325,576,12,0,0,5,Eliminar Vehiculo,LJAIMES_112,Luis Eduardo Jaimes,"<= Elimination Vehicle =>\nIdLinea=10325,\nSer...",14,No se presenta conductor a realizar servicio,0
3,19/05/2026,0:28:48,10273,142,4,0,0,20,Cambiar Conductor,JBERNAL_105,JENNY BERNAL ALEXANDRA BERNAL,com.lgcns.fms.oprt.schedule.driver.model.Chang...,0,Motivo no definido.,0
4,19/05/2026,0:29:14,10273,142,15,0,0,20,Cambiar Conductor,JBERNAL_105,JENNY BERNAL ALEXANDRA BERNAL,com.lgcns.fms.oprt.schedule.driver.model.Chang...,0,Motivo no definido.,0


In [169]:
#Crear copias de acciones de regulación SAE

acciones_con_acción_5 = acciones_SAE # Acción de eliminación
acciones_con_acción_4 = acciones_SAE # Acción de introducir coche
acciones_con_acción_36 = acciones_SAE # Acción de cambio de coche

In [170]:
# Acciónes de regulación igual a 5 y Motivo igual a congestión vehicular

# Filtrar donde Accion = 5 y DescripcionMotivo = "Congestión vehicular"
acciones_con_acción_5 = acciones_con_acción_5[
    (acciones_con_acción_5['Id Acción'] == 5) &
    (acciones_con_acción_5['Motivo'] == 'Congestion vehicular')
]

acciones_con_acción_5.head()

,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,Nombre de Usuario,Parámetros,Id Motivo,Motivo,Reversada por
700,19/05/2026,9:33:26,10266,DD204,3,0,0,5,Eliminar Vehiculo,RROMERO_105,Raul Fabian Romero Mican,"<= Elimination Vehicle =>\nIdLinea=10266,\nSer...",5,Congestion vehicular,0
736,19/05/2026,9:54:31,10551,KL307,9,0,0,5,Eliminar Vehiculo,YHERNANDEZ_105,YERLY HERNANDEZ IRENE ARIZA,"<= Elimination Vehicle =>\nIdLinea=10551,\nSer...",5,Congestion vehicular,0


In [171]:
# Extraer el valor entre comillas después de Servicio=
acciones_con_acción_5['ServBus Retoma'] = (
    acciones_con_acción_5['Parámetros']
    .str.extract(r'ServicioBus=([^\n]+)')
)

acciones_con_acción_5.head()

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9404\1738585510.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acciones_con_acción_5['ServBus Retoma'] = (


,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,Nombre de Usuario,Parámetros,Id Motivo,Motivo,Reversada por,ServBus Retoma
700,19/05/2026,9:33:26,10266,DD204,3,0,0,5,Eliminar Vehiculo,RROMERO_105,Raul Fabian Romero Mican,"<= Elimination Vehicle =>\nIdLinea=10266,\nSer...",5,Congestion vehicular,0,"CE0630003,"
736,19/05/2026,9:54:31,10551,KL307,9,0,0,5,Eliminar Vehiculo,YHERNANDEZ_105,YERLY HERNANDEZ IRENE ARIZA,"<= Elimination Vehicle =>\nIdLinea=10551,\nSer...",5,Congestion vehicular,0,"CE1BA0009,"


In [172]:
# Extraer el valor entre comillas después de Servicio=

# Extraer el valor entre comillas después de Servicio=
acciones_con_acción_5['ServBus'] = (
    acciones_con_acción_5['Parámetros']
    .str.extract(r'ServicioBus=([^\n]+)')
)

acciones_con_acción_5.head()

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9404\2162346313.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acciones_con_acción_5['ServBus'] = (


,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,Nombre de Usuario,Parámetros,Id Motivo,Motivo,Reversada por,ServBus Retoma,ServBus
700,19/05/2026,9:33:26,10266,DD204,3,0,0,5,Eliminar Vehiculo,RROMERO_105,Raul Fabian Romero Mican,"<= Elimination Vehicle =>\nIdLinea=10266,\nSer...",5,Congestion vehicular,0,"CE0630003,","CE0630003,"
736,19/05/2026,9:54:31,10551,KL307,9,0,0,5,Eliminar Vehiculo,YHERNANDEZ_105,YERLY HERNANDEZ IRENE ARIZA,"<= Elimination Vehicle =>\nIdLinea=10551,\nSer...",5,Congestion vehicular,0,"CE1BA0009,","CE1BA0009,"


In [173]:
acciones_con_acción_5['CocheOrig'] = acciones_con_acción_5['Tabla']

acciones_con_acción_5.head()

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9404\1640137476.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acciones_con_acción_5['CocheOrig'] = acciones_con_acción_5['Tabla']


,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,Nombre de Usuario,Parámetros,Id Motivo,Motivo,Reversada por,ServBus Retoma,ServBus,CocheOrig
700,19/05/2026,9:33:26,10266,DD204,3,0,0,5,Eliminar Vehiculo,RROMERO_105,Raul Fabian Romero Mican,"<= Elimination Vehicle =>\nIdLinea=10266,\nSer...",5,Congestion vehicular,0,"CE0630003,","CE0630003,",3
736,19/05/2026,9:54:31,10551,KL307,9,0,0,5,Eliminar Vehiculo,YHERNANDEZ_105,YERLY HERNANDEZ IRENE ARIZA,"<= Elimination Vehicle =>\nIdLinea=10551,\nSer...",5,Congestion vehicular,0,"CE1BA0009,","CE1BA0009,",9


In [174]:
acciones_con_acción_5['CocheNuevo'] = acciones_con_acción_5['Tabla']

acciones_con_acción_5.head()

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9404\2957074855.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acciones_con_acción_5['CocheNuevo'] = acciones_con_acción_5['Tabla']


,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,Nombre de Usuario,Parámetros,Id Motivo,Motivo,Reversada por,ServBus Retoma,ServBus,CocheOrig,CocheNuevo
700,19/05/2026,9:33:26,10266,DD204,3,0,0,5,Eliminar Vehiculo,RROMERO_105,Raul Fabian Romero Mican,"<= Elimination Vehicle =>\nIdLinea=10266,\nSer...",5,Congestion vehicular,0,"CE0630003,","CE0630003,",3,3
736,19/05/2026,9:54:31,10551,KL307,9,0,0,5,Eliminar Vehiculo,YHERNANDEZ_105,YERLY HERNANDEZ IRENE ARIZA,"<= Elimination Vehicle =>\nIdLinea=10551,\nSer...",5,Congestion vehicular,0,"CE1BA0009,","CE1BA0009,",9,9


In [175]:
# Extraer RutaSAE
acciones_con_acción_5['RutaSAE'] = (
    acciones_con_acción_5['Parámetros']
    .str.extract(r'IdRutaDesde=([^\n]+)')
)

# Extraer ViajeIni
acciones_con_acción_5['ViajeIni'] = (
    acciones_con_acción_5['Parámetros']
    .str.extract(r'IdViajeDesde=([^\n]+)')
)

acciones_con_acción_5[
    ['RutaSAE', 'ViajeIni']
].head()

acciones_con_acción_5.head()

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9404\2018998590.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acciones_con_acción_5['RutaSAE'] = (
C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9404\2018998590.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acciones_con_acción_5['ViajeIni'] = (


,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,...,Parámetros,Id Motivo,Motivo,Reversada por,ServBus Retoma,ServBus,CocheOrig,CocheNuevo,RutaSAE,ViajeIni
700,19/05/2026,9:33:26,10266,DD204,3,0,0,5,Eliminar Vehiculo,RROMERO_105,...,"<= Elimination Vehicle =>\nIdLinea=10266,\nSer...",5,Congestion vehicular,0,"CE0630003,","CE0630003,",3,3,"10533,","6,"
736,19/05/2026,9:54:31,10551,KL307,9,0,0,5,Eliminar Vehiculo,YHERNANDEZ_105,...,"<= Elimination Vehicle =>\nIdLinea=10551,\nSer...",5,Congestion vehicular,0,"CE1BA0009,","CE1BA0009,",9,9,"12856,","2,"


In [176]:
# Patrón: busca la etiqueta <Hasta ...> y captura SOLO sus atributos
# Extraer ViajeFin
acciones_con_acción_5['ViajeFin'] = (
    acciones_con_acción_5['Parámetros']
    .str.extract(r'IdViajeHasta=([^\n]+)')
)

acciones_con_acción_5.head()

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9404\2325692457.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acciones_con_acción_5['ViajeFin'] = (


,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,...,Id Motivo,Motivo,Reversada por,ServBus Retoma,ServBus,CocheOrig,CocheNuevo,RutaSAE,ViajeIni,ViajeFin
700,19/05/2026,9:33:26,10266,DD204,3,0,0,5,Eliminar Vehiculo,RROMERO_105,...,5,Congestion vehicular,0,"CE0630003,","CE0630003,",3,3,"10533,","6,","8,"
736,19/05/2026,9:54:31,10551,KL307,9,0,0,5,Eliminar Vehiculo,YHERNANDEZ_105,...,5,Congestion vehicular,0,"CE1BA0009,","CE1BA0009,",9,9,"12856,","2,","4,"


In [177]:
# Acciónes de regulación igual a 4

acciones_con_acción_4 = acciones_con_acción_4[
    (acciones_con_acción_4['Id Acción'] == 4)
]

acciones_con_acción_4.head()

,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,Nombre de Usuario,Parámetros,Id Motivo,Motivo,Reversada por


In [178]:
# Extraer el valor entre comillas después de Servicio=

acciones_con_acción_4['ServBus Retoma'] = (
    acciones_con_acción_4['Parámetros']
    .str.extract(r'vehServId=([^\n]+)')
)

acciones_con_acción_4.head()

,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,Nombre de Usuario,Parámetros,Id Motivo,Motivo,Reversada por,ServBus Retoma


In [179]:
# Extraer el valor entre comillas después de Referencia Servicio=

acciones_con_acción_4['ServBus'] = (
    acciones_con_acción_4['Parámetros']
    .str.extract(r'vehServId=([^\n]+)')
)

acciones_con_acción_4.head()

,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,Nombre de Usuario,Parámetros,Id Motivo,Motivo,Reversada por,ServBus Retoma,ServBus


In [180]:
# Extraer el valor entre comillas después de Coche=

acciones_con_acción_4['CocheOrig'] = (
    acciones_con_acción_4['Parámetros']
    .str.extract(r'lineServId=([^\n]+)')
)

acciones_con_acción_4.head()

,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,Nombre de Usuario,Parámetros,Id Motivo,Motivo,Reversada por,ServBus Retoma,ServBus,CocheOrig


In [181]:
#Nuevo coche, duplicar la columna de Coche
acciones_con_acción_4['CocheNuevo'] = acciones_con_acción_4['Tabla']

acciones_con_acción_4.head()

,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,Nombre de Usuario,Parámetros,Id Motivo,Motivo,Reversada por,ServBus Retoma,ServBus,CocheOrig,CocheNuevo


In [182]:
# Extraer RutaSAE
acciones_con_acción_4['RutaSAE'] = (
    acciones_con_acción_4['Parámetros']
    .str.extract(r'lineServId=([^\n]+)')
)

# Extraer ViajeIni

acciones_con_acción_4['ViajeIni'] = (
    acciones_con_acción_4['Parámetros']
    .str.extract(r' fromServTripSeq=([^\n]+)')
)

acciones_con_acción_4.head()

,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,...,Parámetros,Id Motivo,Motivo,Reversada por,ServBus Retoma,ServBus,CocheOrig,CocheNuevo,RutaSAE,ViajeIni


In [183]:
# Patrón: busca la etiqueta <Hasta ...> y captura SOLO sus atributos
acciones_con_acción_4['ViajeFin'] = (
    acciones_con_acción_4['Parámetros']
    .str.extract(r'toServTripSeq=([^\n]+)')
)

acciones_con_acción_4['HoraIniTeor'] = (
    acciones_con_acción_4['Parámetros']
    .str.extract(r'fromNodeEventSecond=([^\n]+)')
)

acciones_con_acción_4['HoraFinTeor'] = (
    acciones_con_acción_4['Parámetros']
    .str.extract(r'toNodeEventSecond=([^\n]+)')
)

acciones_con_acción_4.head()

,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,...,Reversada por,ServBus Retoma,ServBus,CocheOrig,CocheNuevo,RutaSAE,ViajeIni,ViajeFin,HoraIniTeor,HoraFinTeor


In [184]:
# Convertir a numérico
acciones_con_acción_4['HoraIniTeor'] = pd.to_numeric(
    acciones_con_acción_4['HoraIniTeor'],
    errors='coerce'
)

acciones_con_acción_4['HoraFinTeor'] = pd.to_numeric(
    acciones_con_acción_4['HoraFinTeor'],
    errors='coerce'
)

# Convertir segundos a HH:MM:SS
acciones_con_acción_4['HoraIniTeor'] = (
    pd.to_timedelta(
        acciones_con_acción_4['HoraIniTeor'],
        unit='s'
    )
    .astype(str)
    .str.split().str[-1]
)

acciones_con_acción_4['HoraFinTeor'] = (
    pd.to_timedelta(
        acciones_con_acción_4['HoraFinTeor'],
        unit='s'
    )
    .astype(str)
    .str.split().str[-1]
)

acciones_con_acción_4.head()

,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,...,Reversada por,ServBus Retoma,ServBus,CocheOrig,CocheNuevo,RutaSAE,ViajeIni,ViajeFin,HoraIniTeor,HoraFinTeor


In [185]:
# Acciónes de regulación igual a 36

acciones_con_acción_36 = acciones_con_acción_36[
    (acciones_con_acción_36['Id Acción'] == 36)
]

acciones_con_acción_36.head()

,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,Nombre de Usuario,Parámetros,Id Motivo,Motivo,Reversada por
324,19/05/2026,6:31:11,10331,12,3,Z50-4392,504392,36,Retomar Viajes,RROMERO_105,Raul Fabian Romero Mican,"<= Change Vehicle =>\nIdLinea=10331,\nServBusR...",26,Retoma de viaje,0
332,19/05/2026,6:34:59,10310,C101,4,Z50-2073,502073,36,Retomar Viajes,DPATARROYO_105,DORIS PATARROYO EDITH GORDO,"<= Change Vehicle =>\nIdLinea=10310,\nServBusR...",26,Retoma de viaje,0
345,19/05/2026,6:44:30,10264,614,33,Z50-4303,504303,36,Retomar Viajes,JGUZMAN_105,JINNETH GUZMAN ALEXANDRA HERNANDEZ,"<= Change Vehicle =>\nIdLinea=10264,\nServBusR...",26,Retoma de viaje,0
365,19/05/2026,6:55:34,10194,539,3,Z50-7042,507042,36,Retomar Viajes,ATELLEZ_105,ADRIANA TELLEZ PATRICIA VARELA,"<= Change Vehicle =>\nIdLinea=10194,\nServBusR...",26,Retoma de viaje,0
368,19/05/2026,7:00:42,10261,466,7,Z50-7000,507000,36,Retomar Viajes,JDURAN_105,JIMMY ALEXANDER CARABALLO DIAZ,"<= Change Vehicle =>\nIdLinea=10261,\nServBusR...",26,Retoma de viaje,0


In [186]:
# Extraer el valor entre comillas después de Servicio=
acciones_con_acción_36['ServBus Retoma'] = (
    acciones_con_acción_36['Parámetros']
    .str.extract(r'ServBusNuevo=([^\n]+)')
)

acciones_con_acción_36.head()

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9404\3646876892.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acciones_con_acción_36['ServBus Retoma'] = (


,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,Nombre de Usuario,Parámetros,Id Motivo,Motivo,Reversada por,ServBus Retoma
324,19/05/2026,6:31:11,10331,12,3,Z50-4392,504392,36,Retomar Viajes,RROMERO_105,Raul Fabian Romero Mican,"<= Change Vehicle =>\nIdLinea=10331,\nServBusR...",26,Retoma de viaje,0,"CE12BG003,"
332,19/05/2026,6:34:59,10310,C101,4,Z50-2073,502073,36,Retomar Viajes,DPATARROYO_105,DORIS PATARROYO EDITH GORDO,"<= Change Vehicle =>\nIdLinea=10310,\nServBusR...",26,Retoma de viaje,0,"CE163G010,"
345,19/05/2026,6:44:30,10264,614,33,Z50-4303,504303,36,Retomar Viajes,JGUZMAN_105,JINNETH GUZMAN ALEXANDRA HERNANDEZ,"<= Change Vehicle =>\nIdLinea=10264,\nServBusR...",26,Retoma de viaje,0,"CE12DG033,"
365,19/05/2026,6:55:34,10194,539,3,Z50-7042,507042,36,Retomar Viajes,ATELLEZ_105,ADRIANA TELLEZ PATRICIA VARELA,"<= Change Vehicle =>\nIdLinea=10194,\nServBusR...",26,Retoma de viaje,0,"CE169G006,"
368,19/05/2026,7:00:42,10261,466,7,Z50-7000,507000,36,Retomar Viajes,JDURAN_105,JIMMY ALEXANDER CARABALLO DIAZ,"<= Change Vehicle =>\nIdLinea=10261,\nServBusR...",26,Retoma de viaje,0,"CE1B6G007,"


In [187]:
# Extraer el valor entre comillas después de Referencia Servicio=
acciones_con_acción_36['ServBus'] = (
    acciones_con_acción_36['Parámetros']
    .str.extract(r'ServBusRef=([^\n]+)')
)

acciones_con_acción_36.head()

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9404\3471527843.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acciones_con_acción_36['ServBus'] = (


,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,Nombre de Usuario,Parámetros,Id Motivo,Motivo,Reversada por,ServBus Retoma,ServBus
324,19/05/2026,6:31:11,10331,12,3,Z50-4392,504392,36,Retomar Viajes,RROMERO_105,Raul Fabian Romero Mican,"<= Change Vehicle =>\nIdLinea=10331,\nServBusR...",26,Retoma de viaje,0,"CE12BG003,","CE12B0003,"
332,19/05/2026,6:34:59,10310,C101,4,Z50-2073,502073,36,Retomar Viajes,DPATARROYO_105,DORIS PATARROYO EDITH GORDO,"<= Change Vehicle =>\nIdLinea=10310,\nServBusR...",26,Retoma de viaje,0,"CE163G010,","CE1630010,"
345,19/05/2026,6:44:30,10264,614,33,Z50-4303,504303,36,Retomar Viajes,JGUZMAN_105,JINNETH GUZMAN ALEXANDRA HERNANDEZ,"<= Change Vehicle =>\nIdLinea=10264,\nServBusR...",26,Retoma de viaje,0,"CE12DG033,","CE12D0033,"
365,19/05/2026,6:55:34,10194,539,3,Z50-7042,507042,36,Retomar Viajes,ATELLEZ_105,ADRIANA TELLEZ PATRICIA VARELA,"<= Change Vehicle =>\nIdLinea=10194,\nServBusR...",26,Retoma de viaje,0,"CE169G006,","CE1690006,"
368,19/05/2026,7:00:42,10261,466,7,Z50-7000,507000,36,Retomar Viajes,JDURAN_105,JIMMY ALEXANDER CARABALLO DIAZ,"<= Change Vehicle =>\nIdLinea=10261,\nServBusR...",26,Retoma de viaje,0,"CE1B6G007,","CE1B60007,"


In [188]:
acciones_con_acción_36['CocheNuevo'] = acciones_con_acción_36['Tabla']

acciones_con_acción_36.head()

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9404\2810430639.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acciones_con_acción_36['CocheNuevo'] = acciones_con_acción_36['Tabla']


,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,Nombre de Usuario,Parámetros,Id Motivo,Motivo,Reversada por,ServBus Retoma,ServBus,CocheNuevo
324,19/05/2026,6:31:11,10331,12,3,Z50-4392,504392,36,Retomar Viajes,RROMERO_105,Raul Fabian Romero Mican,"<= Change Vehicle =>\nIdLinea=10331,\nServBusR...",26,Retoma de viaje,0,"CE12BG003,","CE12B0003,",3
332,19/05/2026,6:34:59,10310,C101,4,Z50-2073,502073,36,Retomar Viajes,DPATARROYO_105,DORIS PATARROYO EDITH GORDO,"<= Change Vehicle =>\nIdLinea=10310,\nServBusR...",26,Retoma de viaje,0,"CE163G010,","CE1630010,",4
345,19/05/2026,6:44:30,10264,614,33,Z50-4303,504303,36,Retomar Viajes,JGUZMAN_105,JINNETH GUZMAN ALEXANDRA HERNANDEZ,"<= Change Vehicle =>\nIdLinea=10264,\nServBusR...",26,Retoma de viaje,0,"CE12DG033,","CE12D0033,",33
365,19/05/2026,6:55:34,10194,539,3,Z50-7042,507042,36,Retomar Viajes,ATELLEZ_105,ADRIANA TELLEZ PATRICIA VARELA,"<= Change Vehicle =>\nIdLinea=10194,\nServBusR...",26,Retoma de viaje,0,"CE169G006,","CE1690006,",3
368,19/05/2026,7:00:42,10261,466,7,Z50-7000,507000,36,Retomar Viajes,JDURAN_105,JIMMY ALEXANDER CARABALLO DIAZ,"<= Change Vehicle =>\nIdLinea=10261,\nServBusR...",26,Retoma de viaje,0,"CE1B6G007,","CE1B60007,",7


In [189]:
# Extraer el valor entre comillas después de Coche=
acciones_con_acción_36['CocheOrig'] = (
    acciones_con_acción_36['Parámetros']
    .str.extract(r'TablaRef=([^\n]+)')
)

acciones_con_acción_36.head()

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9404\3785817740.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acciones_con_acción_36['CocheOrig'] = (


,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,Nombre de Usuario,Parámetros,Id Motivo,Motivo,Reversada por,ServBus Retoma,ServBus,CocheNuevo,CocheOrig
324,19/05/2026,6:31:11,10331,12,3,Z50-4392,504392,36,Retomar Viajes,RROMERO_105,Raul Fabian Romero Mican,"<= Change Vehicle =>\nIdLinea=10331,\nServBusR...",26,Retoma de viaje,0,"CE12BG003,","CE12B0003,",3,"3,"
332,19/05/2026,6:34:59,10310,C101,4,Z50-2073,502073,36,Retomar Viajes,DPATARROYO_105,DORIS PATARROYO EDITH GORDO,"<= Change Vehicle =>\nIdLinea=10310,\nServBusR...",26,Retoma de viaje,0,"CE163G010,","CE1630010,",4,"4,"
345,19/05/2026,6:44:30,10264,614,33,Z50-4303,504303,36,Retomar Viajes,JGUZMAN_105,JINNETH GUZMAN ALEXANDRA HERNANDEZ,"<= Change Vehicle =>\nIdLinea=10264,\nServBusR...",26,Retoma de viaje,0,"CE12DG033,","CE12D0033,",33,"33,"
365,19/05/2026,6:55:34,10194,539,3,Z50-7042,507042,36,Retomar Viajes,ATELLEZ_105,ADRIANA TELLEZ PATRICIA VARELA,"<= Change Vehicle =>\nIdLinea=10194,\nServBusR...",26,Retoma de viaje,0,"CE169G006,","CE1690006,",3,"3,"
368,19/05/2026,7:00:42,10261,466,7,Z50-7000,507000,36,Retomar Viajes,JDURAN_105,JIMMY ALEXANDER CARABALLO DIAZ,"<= Change Vehicle =>\nIdLinea=10261,\nServBusR...",26,Retoma de viaje,0,"CE1B6G007,","CE1B60007,",7,"7,"


In [190]:
# Extraer RutaSAE
acciones_con_acción_36['RutaSAE'] = (
    acciones_con_acción_36['Parámetros']
    .str.extract(r'IdRutaDesdeRef=([^\n]+)')
)

# Extraer ViajeIni
acciones_con_acción_36['ViajeIni'] = (
    acciones_con_acción_36['Parámetros']
    .str.extract(r'IdViajeDesdeRef=([^\n]+)')
)
acciones_con_acción_36.head()

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9404\3411046025.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acciones_con_acción_36['RutaSAE'] = (
C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9404\3411046025.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acciones_con_acción_36['ViajeIni'] = (


,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,...,Parámetros,Id Motivo,Motivo,Reversada por,ServBus Retoma,ServBus,CocheNuevo,CocheOrig,RutaSAE,ViajeIni
324,19/05/2026,6:31:11,10331,12,3,Z50-4392,504392,36,Retomar Viajes,RROMERO_105,...,"<= Change Vehicle =>\nIdLinea=10331,\nServBusR...",26,Retoma de viaje,0,"CE12BG003,","CE12B0003,",3,"3,","10632,","3,"
332,19/05/2026,6:34:59,10310,C101,4,Z50-2073,502073,36,Retomar Viajes,DPATARROYO_105,...,"<= Change Vehicle =>\nIdLinea=10310,\nServBusR...",26,Retoma de viaje,0,"CE163G010,","CE1630010,",4,"4,","12777,","3,"
345,19/05/2026,6:44:30,10264,614,33,Z50-4303,504303,36,Retomar Viajes,JGUZMAN_105,...,"<= Change Vehicle =>\nIdLinea=10264,\nServBusR...",26,Retoma de viaje,0,"CE12DG033,","CE12D0033,",33,"33,","12328,","3,"
365,19/05/2026,6:55:34,10194,539,3,Z50-7042,507042,36,Retomar Viajes,ATELLEZ_105,...,"<= Change Vehicle =>\nIdLinea=10194,\nServBusR...",26,Retoma de viaje,0,"CE169G006,","CE1690006,",3,"3,","12786,","3,"
368,19/05/2026,7:00:42,10261,466,7,Z50-7000,507000,36,Retomar Viajes,JDURAN_105,...,"<= Change Vehicle =>\nIdLinea=10261,\nServBusR...",26,Retoma de viaje,0,"CE1B6G007,","CE1B60007,",7,"7,","13058,","3,"


In [191]:
#Hasta ruta SAE fms
acciones_con_acción_36['RutaSAE_h'] = (
    acciones_con_acción_36['Parámetros']
    .str.extract(r'IdRutaHastaRef=([^\n]+)')
)

acciones_con_acción_36.head()

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9404\3328309916.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acciones_con_acción_36['RutaSAE_h'] = (


,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,...,Id Motivo,Motivo,Reversada por,ServBus Retoma,ServBus,CocheNuevo,CocheOrig,RutaSAE,ViajeIni,RutaSAE_h
324,19/05/2026,6:31:11,10331,12,3,Z50-4392,504392,36,Retomar Viajes,RROMERO_105,...,26,Retoma de viaje,0,"CE12BG003,","CE12B0003,",3,"3,","10632,","3,","10632,"
332,19/05/2026,6:34:59,10310,C101,4,Z50-2073,502073,36,Retomar Viajes,DPATARROYO_105,...,26,Retoma de viaje,0,"CE163G010,","CE1630010,",4,"4,","12777,","3,","12778,"
345,19/05/2026,6:44:30,10264,614,33,Z50-4303,504303,36,Retomar Viajes,JGUZMAN_105,...,26,Retoma de viaje,0,"CE12DG033,","CE12D0033,",33,"33,","12328,","3,","12328,"
365,19/05/2026,6:55:34,10194,539,3,Z50-7042,507042,36,Retomar Viajes,ATELLEZ_105,...,26,Retoma de viaje,0,"CE169G006,","CE1690006,",3,"3,","12786,","3,","12785,"
368,19/05/2026,7:00:42,10261,466,7,Z50-7000,507000,36,Retomar Viajes,JDURAN_105,...,26,Retoma de viaje,0,"CE1B6G007,","CE1B60007,",7,"7,","13058,","3,","13058,"


In [192]:
# Patrón: busca la etiqueta <Hasta ...> y captura SOLO sus atributos
acciones_con_acción_36['ViajeFin'] = (
    acciones_con_acción_36['Parámetros']
    .str.extract(r'IdViajeHastaRef=([^\n]+)')
)

acciones_con_acción_36.head()

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9404\1601864151.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acciones_con_acción_36['ViajeFin'] = (


,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,...,Motivo,Reversada por,ServBus Retoma,ServBus,CocheNuevo,CocheOrig,RutaSAE,ViajeIni,RutaSAE_h,ViajeFin
324,19/05/2026,6:31:11,10331,12,3,Z50-4392,504392,36,Retomar Viajes,RROMERO_105,...,Retoma de viaje,0,"CE12BG003,","CE12B0003,",3,"3,","10632,","3,","10632,","3,"
332,19/05/2026,6:34:59,10310,C101,4,Z50-2073,502073,36,Retomar Viajes,DPATARROYO_105,...,Retoma de viaje,0,"CE163G010,","CE1630010,",4,"4,","12777,","3,","12778,","4,"
345,19/05/2026,6:44:30,10264,614,33,Z50-4303,504303,36,Retomar Viajes,JGUZMAN_105,...,Retoma de viaje,0,"CE12DG033,","CE12D0033,",33,"33,","12328,","3,","12328,","3,"
365,19/05/2026,6:55:34,10194,539,3,Z50-7042,507042,36,Retomar Viajes,ATELLEZ_105,...,Retoma de viaje,0,"CE169G006,","CE1690006,",3,"3,","12786,","3,","12785,","4,"
368,19/05/2026,7:00:42,10261,466,7,Z50-7000,507000,36,Retomar Viajes,JDURAN_105,...,Retoma de viaje,0,"CE1B6G007,","CE1B60007,",7,"7,","13058,","3,","13058,","3,"


In [193]:
acciones_con_acción_36.rename(columns={'ï»¿Fecha': 'Fecha'}, inplace=True)

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9404\1364010224.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acciones_con_acción_36.rename(columns={'ï»¿Fecha': 'Fecha'}, inplace=True)


In [194]:
acciones_final = pd.concat([acciones_con_acción_4,acciones_con_acción_5, acciones_con_acción_36], ignore_index=True)

acciones_final.head()


,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,...,ServBus Retoma,ServBus,CocheOrig,CocheNuevo,RutaSAE,ViajeIni,ViajeFin,HoraIniTeor,HoraFinTeor,RutaSAE_h
0,19/05/2026,9:33:26,10266,DD204,3,0,0,5,Eliminar Vehiculo,RROMERO_105,...,"CE0630003,","CE0630003,",3,3,"10533,","6,","8,",NaN,NaN,NaN
1,19/05/2026,9:54:31,10551,KL307,9,0,0,5,Eliminar Vehiculo,YHERNANDEZ_105,...,"CE1BA0009,","CE1BA0009,",9,9,"12856,","2,","4,",NaN,NaN,NaN
2,19/05/2026,6:31:11,10331,12,3,Z50-4392,504392,36,Retomar Viajes,RROMERO_105,...,"CE12BG003,","CE12B0003,","3,",3,"10632,","3,","3,",NaN,NaN,"10632,"
3,19/05/2026,6:34:59,10310,C101,4,Z50-2073,502073,36,Retomar Viajes,DPATARROYO_105,...,"CE163G010,","CE1630010,","4,",4,"12777,","3,","4,",NaN,NaN,"12778,"
4,19/05/2026,6:44:30,10264,614,33,Z50-4303,504303,36,Retomar Viajes,JGUZMAN_105,...,"CE12DG033,","CE12D0033,","33,",33,"12328,","3,","3,",NaN,NaN,"12328,"


In [195]:
# Diccionario ServBus → ServBus Retoma
serv_a_retoma = dict(zip(acciones_final['ServBus'], acciones_final['ServBus Retoma']))

# Función para encontrar el "raíz" de la cadena
def encontrar_raiz(servicio):
    actual = servicio
    visitados = set()
    while actual in serv_a_retoma.values() and actual not in visitados:
        visitados.add(actual)
        padre = acciones_final.loc[acciones_final['ServBus Retoma'] == actual, 'ServBus']
        if not padre.empty:
            actual = padre.values[0]
        else:
            break
    return actual

# Crear la nueva columna directamente en acciones_final
acciones_final['Raiz'] = acciones_final['ServBus'].apply(encontrar_raiz)

acciones_final.head()

,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,...,ServBus,CocheOrig,CocheNuevo,RutaSAE,ViajeIni,ViajeFin,HoraIniTeor,HoraFinTeor,RutaSAE_h,Raiz
0,19/05/2026,9:33:26,10266,DD204,3,0,0,5,Eliminar Vehiculo,RROMERO_105,...,"CE0630003,",3,3,"10533,","6,","8,",NaN,NaN,NaN,"CE0630003,"
1,19/05/2026,9:54:31,10551,KL307,9,0,0,5,Eliminar Vehiculo,YHERNANDEZ_105,...,"CE1BA0009,",9,9,"12856,","2,","4,",NaN,NaN,NaN,"CE1BA0009,"
2,19/05/2026,6:31:11,10331,12,3,Z50-4392,504392,36,Retomar Viajes,RROMERO_105,...,"CE12B0003,","3,",3,"10632,","3,","3,",NaN,NaN,"10632,","CE12B0003,"
3,19/05/2026,6:34:59,10310,C101,4,Z50-2073,502073,36,Retomar Viajes,DPATARROYO_105,...,"CE1630010,","4,",4,"12777,","3,","4,",NaN,NaN,"12778,","CE1630010,"
4,19/05/2026,6:44:30,10264,614,33,Z50-4303,504303,36,Retomar Viajes,JGUZMAN_105,...,"CE12D0033,","33,",33,"12328,","3,","3,",NaN,NaN,"12328,","CE12D0033,"


In [196]:
# --- NORMALIZACIÓN BÁSICA ---
for col in ['ServBus', 'ServBus Retoma']:
    if col in acciones_final.columns:
        acciones_final[col] = (acciones_final[col]
                               .astype(str).str.strip().str.upper()
                               .replace({'NAN': pd.NA}))

# Si no existe 'Servicio', lo creamos tomando el valor actual de ServBus
if 'Servicio' not in acciones_final.columns:
    acciones_final['Servicio'] = acciones_final['ServBus']
else:
    acciones_final['Servicio'] = (acciones_final['Servicio']
                                  .astype(str).str.strip().str.upper()
                                  .replace({'NAN': pd.NA}))

# --- MAPA: para cada 'ServBus Retoma', cuál es el 'ServBus' raíz según Accion 4/36 ---
mask_ref = acciones_final['Id Acción'].isin([4, 36])

# Elegimos el primer (o único) 'ServBus' visto por cada 'ServBus Retoma' en acciones 4/36
root_map = (acciones_final.loc[mask_ref, ['ServBus Retoma', 'ServBus']]
            .dropna()
            .drop_duplicates(subset=['ServBus Retoma'])    # si hay varios, toma el primero
            .set_index('ServBus Retoma')['ServBus'])

# --- ACTUALIZAR SOLO FILAS CON Accion == 5 ---
mask_a5 = acciones_final['Id Acción'].eq(5)

# Valor raíz que corresponde al 'ServBus Retoma' de la fila con Accion 5
raiz_para_a5 = acciones_final.loc[mask_a5, 'ServBus Retoma'].map(root_map)

# Escribimos el raíz solo cuando exista mapeo (no pisamos otros casos)
acciones_final.loc[mask_a5 & raiz_para_a5.notna(), 'Servicio'] = raiz_para_a5.loc[raiz_para_a5.notna()]


acciones_final.head()

,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,...,CocheOrig,CocheNuevo,RutaSAE,ViajeIni,ViajeFin,HoraIniTeor,HoraFinTeor,RutaSAE_h,Raiz,Servicio
0,19/05/2026,9:33:26,10266,DD204,3,0,0,5,Eliminar Vehiculo,RROMERO_105,...,3,3,"10533,","6,","8,",NaN,NaN,NaN,"CE0630003,","CE0630003,"
1,19/05/2026,9:54:31,10551,KL307,9,0,0,5,Eliminar Vehiculo,YHERNANDEZ_105,...,9,9,"12856,","2,","4,",NaN,NaN,NaN,"CE1BA0009,","CE1BA0009,"
2,19/05/2026,6:31:11,10331,12,3,Z50-4392,504392,36,Retomar Viajes,RROMERO_105,...,"3,",3,"10632,","3,","3,",NaN,NaN,"10632,","CE12B0003,","CE12B0003,"
3,19/05/2026,6:34:59,10310,C101,4,Z50-2073,502073,36,Retomar Viajes,DPATARROYO_105,...,"4,",4,"12777,","3,","4,",NaN,NaN,"12778,","CE1630010,","CE1630010,"
4,19/05/2026,6:44:30,10264,614,33,Z50-4303,504303,36,Retomar Viajes,JGUZMAN_105,...,"33,",33,"12328,","3,","3,",NaN,NaN,"12328,","CE12D0033,","CE12D0033,"


In [197]:
# Crear columna 'Final' según la condición solicitada
acciones_final['Final'] = acciones_final.apply(
    lambda x: x['Servicio'] if x['Id Acción'] == 5 else x['Raiz'],
    axis=1
)

acciones_final.head()

,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,...,CocheNuevo,RutaSAE,ViajeIni,ViajeFin,HoraIniTeor,HoraFinTeor,RutaSAE_h,Raiz,Servicio,Final
0,19/05/2026,9:33:26,10266,DD204,3,0,0,5,Eliminar Vehiculo,RROMERO_105,...,3,"10533,","6,","8,",NaN,NaN,NaN,"CE0630003,","CE0630003,","CE0630003,"
1,19/05/2026,9:54:31,10551,KL307,9,0,0,5,Eliminar Vehiculo,YHERNANDEZ_105,...,9,"12856,","2,","4,",NaN,NaN,NaN,"CE1BA0009,","CE1BA0009,","CE1BA0009,"
2,19/05/2026,6:31:11,10331,12,3,Z50-4392,504392,36,Retomar Viajes,RROMERO_105,...,3,"10632,","3,","3,",NaN,NaN,"10632,","CE12B0003,","CE12B0003,","CE12B0003,"
3,19/05/2026,6:34:59,10310,C101,4,Z50-2073,502073,36,Retomar Viajes,DPATARROYO_105,...,4,"12777,","3,","4,",NaN,NaN,"12778,","CE1630010,","CE1630010,","CE1630010,"
4,19/05/2026,6:44:30,10264,614,33,Z50-4303,504303,36,Retomar Viajes,JGUZMAN_105,...,33,"12328,","3,","3,",NaN,NaN,"12328,","CE12D0033,","CE12D0033,","CE12D0033,"


In [198]:
# Llevar el servicio de acciones_final (Final)

def calcular_lin(serv_retoma):
    
    filtro = (
        (acciones_final['ServBus Retoma'] == serv_retoma) 
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not acciones_final.loc[filtro].empty:
        # Obtener el primer valor
        tipo = acciones_final.loc[filtro, 'Final'].iloc[0]
        return tipo if not pd.isna(tipo) else None 
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
eventos_SAE['Servicio_orig'] = eventos_SAE.apply(
    lambda row: calcular_lin(
        row['VEH_SERV_ID']
    ),
    axis=1
)
eventos_SAE.head()

,APPLY_DATE,VEH_SERV_ID,SERV_TRIP_SEQ,VEH_REGISTR_NUM,EVENT_DATETIME,START_DATETIME,END_DATETIME,START_VEH_LAT,START_VEH_LON,END_VEH_LAT,END_VEH_LON,START_ROUTE_OFFSET_VALUE,END_ROUTE_OFFSET_VALUE,TRIP_END_TYPE_CD,CREAT_USER_ID,CREAT_DATETIME,UPD_USER_ID,UPD_DATETIME,Servicio_orig
0,20260519,AD0045123,1,10132,05:27:19,05:27:19,06:33:41,507235,588724,517432,597536,0,23778.0,NaN,SmartCore,20260519052742,SmartCore,2.026052e+13,None
1,20260519,AD0045123,2,10132,06:35:57,06:35:57,07:34:26,517528,597469,507054,588348,0,24797.0,NaN,SmartCore,20260519063600,SmartCore,2.026052e+13,None
2,20260519,AD0066049,1,7028,09:51:23,09:51:23,10:47:05,521962,605199,517528,597472,0,22012.0,NaN,SmartCore,20260519095132,SmartCore,2.026052e+13,None
3,20260519,AD0066050,1,7028,19:45:59,19:45:59,20:42:49,520031,604881,517525,597471,1967,22012.0,NaN,SmartCore,20260519194600,SmartCore,2.026052e+13,None
4,20260519,AD0067054,1,40407,21:03:30,21:03:30,21:30:44,509019,600319,505311,595454,0,24834.0,NaN,SmartCore,20260519210350,SmartCore,2.026052e+13,None


In [199]:
#Servicio final

eventos_SAE['ServBus'] = np.where(
    eventos_SAE['Servicio_orig'].isna() | (eventos_SAE['Servicio_orig'] == ''),
    eventos_SAE['VEH_SERV_ID'],
    eventos_SAE['Servicio_orig']
)

eventos_SAE.head()

,APPLY_DATE,VEH_SERV_ID,SERV_TRIP_SEQ,VEH_REGISTR_NUM,EVENT_DATETIME,START_DATETIME,END_DATETIME,START_VEH_LAT,START_VEH_LON,END_VEH_LAT,END_VEH_LON,START_ROUTE_OFFSET_VALUE,END_ROUTE_OFFSET_VALUE,TRIP_END_TYPE_CD,CREAT_USER_ID,CREAT_DATETIME,UPD_USER_ID,UPD_DATETIME,Servicio_orig,ServBus
0,20260519,AD0045123,1,10132,05:27:19,05:27:19,06:33:41,507235,588724,517432,597536,0,23778.0,NaN,SmartCore,20260519052742,SmartCore,2.026052e+13,None,AD0045123
1,20260519,AD0045123,2,10132,06:35:57,06:35:57,07:34:26,517528,597469,507054,588348,0,24797.0,NaN,SmartCore,20260519063600,SmartCore,2.026052e+13,None,AD0045123
2,20260519,AD0066049,1,7028,09:51:23,09:51:23,10:47:05,521962,605199,517528,597472,0,22012.0,NaN,SmartCore,20260519095132,SmartCore,2.026052e+13,None,AD0066049
3,20260519,AD0066050,1,7028,19:45:59,19:45:59,20:42:49,520031,604881,517525,597471,1967,22012.0,NaN,SmartCore,20260519194600,SmartCore,2.026052e+13,None,AD0066050
4,20260519,AD0067054,1,40407,21:03:30,21:03:30,21:30:44,509019,600319,505311,595454,0,24834.0,NaN,SmartCore,20260519210350,SmartCore,2.026052e+13,None,AD0067054


In [200]:
eventos_SAE['HoraFinal']= eventos_SAE['END_DATETIME']

eventos_SAE.head()

,APPLY_DATE,VEH_SERV_ID,SERV_TRIP_SEQ,VEH_REGISTR_NUM,EVENT_DATETIME,START_DATETIME,END_DATETIME,START_VEH_LAT,START_VEH_LON,END_VEH_LAT,...,START_ROUTE_OFFSET_VALUE,END_ROUTE_OFFSET_VALUE,TRIP_END_TYPE_CD,CREAT_USER_ID,CREAT_DATETIME,UPD_USER_ID,UPD_DATETIME,Servicio_orig,ServBus,HoraFinal
0,20260519,AD0045123,1,10132,05:27:19,05:27:19,06:33:41,507235,588724,517432,...,0,23778.0,NaN,SmartCore,20260519052742,SmartCore,2.026052e+13,None,AD0045123,06:33:41
1,20260519,AD0045123,2,10132,06:35:57,06:35:57,07:34:26,517528,597469,507054,...,0,24797.0,NaN,SmartCore,20260519063600,SmartCore,2.026052e+13,None,AD0045123,07:34:26
2,20260519,AD0066049,1,7028,09:51:23,09:51:23,10:47:05,521962,605199,517528,...,0,22012.0,NaN,SmartCore,20260519095132,SmartCore,2.026052e+13,None,AD0066049,10:47:05
3,20260519,AD0066050,1,7028,19:45:59,19:45:59,20:42:49,520031,604881,517525,...,1967,22012.0,NaN,SmartCore,20260519194600,SmartCore,2.026052e+13,None,AD0066050,20:42:49
4,20260519,AD0067054,1,40407,21:03:30,21:03:30,21:30:44,509019,600319,505311,...,0,24834.0,NaN,SmartCore,20260519210350,SmartCore,2.026052e+13,None,AD0067054,21:30:44


In [201]:
acciones_final['HoraIniTeorica']= acciones_final['HoraIniTeor']

acciones_final.head()

,Fecha,Instante,Id Línea,Línea,Tabla,Código Bus,Número FMS Bus,Id Acción,Descripción de Acción,Id Usuario,...,RutaSAE,ViajeIni,ViajeFin,HoraIniTeor,HoraFinTeor,RutaSAE_h,Raiz,Servicio,Final,HoraIniTeorica
0,19/05/2026,9:33:26,10266,DD204,3,0,0,5,Eliminar Vehiculo,RROMERO_105,...,"10533,","6,","8,",NaN,NaN,NaN,"CE0630003,","CE0630003,","CE0630003,",NaN
1,19/05/2026,9:54:31,10551,KL307,9,0,0,5,Eliminar Vehiculo,YHERNANDEZ_105,...,"12856,","2,","4,",NaN,NaN,NaN,"CE1BA0009,","CE1BA0009,","CE1BA0009,",NaN
2,19/05/2026,6:31:11,10331,12,3,Z50-4392,504392,36,Retomar Viajes,RROMERO_105,...,"10632,","3,","3,",NaN,NaN,"10632,","CE12B0003,","CE12B0003,","CE12B0003,",NaN
3,19/05/2026,6:34:59,10310,C101,4,Z50-2073,502073,36,Retomar Viajes,DPATARROYO_105,...,"12777,","3,","4,",NaN,NaN,"12778,","CE1630010,","CE1630010,","CE1630010,",NaN
4,19/05/2026,6:44:30,10264,614,33,Z50-4303,504303,36,Retomar Viajes,JGUZMAN_105,...,"12328,","3,","3,",NaN,NaN,"12328,","CE12D0033,","CE12D0033,","CE12D0033,",NaN


Revisar los datos los esta arrojando en cero de la columna de horas

In [202]:
# Función para corregir tiempos con "24:00:00"
def fix_time(date_str):
    if '24:' in date_str:
        return date_str.replace('24:', '00:')
    elif '25:' in date_str:
        return date_str.replace('25:', '01:')
    elif '26:' in date_str:
        return date_str.replace('26:', '02:')
    elif '27:' in date_str:
        return date_str.replace('27:', '03:')
    elif '28:' in date_str:
        return date_str.replace('28:', '04:')
    elif '29:' in date_str:
        return date_str.replace('29:', '05:')
    else:
        return date_str

# Función para convertir tiempo a segundos
def time_to_seconds(time_obj):
    return time_obj.hour * 3600 + time_obj.minute * 60 + time_obj.second

# Convertir la columna de tiempo a cadenas
eventos_SAE['HoraFinal'] = eventos_SAE['HoraFinal'].astype(str)
acciones_final['HoraIniTeorica'] = acciones_final['HoraIniTeorica'].astype(str)

# Aplicar la función para corregir los tiempos
eventos_SAE['HoraFinal'] = eventos_SAE['HoraFinal'].apply(fix_time)
acciones_final['HoraIniTeorica'] = acciones_final['HoraIniTeorica'].apply(fix_time)

# Convertir la columna de tiempo a datetime, usando errors='coerce' para manejar errores
eventos_SAE['HoraFinal'] = pd.to_datetime(eventos_SAE['HoraFinal'], format='%H:%M', errors='coerce')
acciones_final['HoraIniTeorica'] = pd.to_datetime(acciones_final['HoraIniTeorica'], format='%H:%M:%S', errors='coerce')

# Eliminar la fecha predeterminada para trabajar solo con la parte de tiempo
eventos_SAE['HoraFinal'] = eventos_SAE['HoraFinal'].dt.time
acciones_final['HoraIniTeorica'] = acciones_final['HoraIniTeorica'].dt.time

# Manejar NaT después de la conversión
eventos_SAE['HoraFinal'] = eventos_SAE['HoraFinal'].apply(lambda x: x if pd.notnull(x) else pd.Timestamp('00:00:00').time())
acciones_final['HoraIniTeorica'] = acciones_final['HoraIniTeorica'].apply(lambda x: x if pd.notnull(x) else pd.Timestamp('00:00:00').time())

# Convertir la columna 'Instante' a segundos
eventos_SAE['HoraFinal_segundos'] = eventos_SAE['HoraFinal'].apply(time_to_seconds).astype(int)
acciones_final['HoraIniTeorica_segundos'] = acciones_final['HoraIniTeorica'].apply(time_to_seconds).astype(int)

eventos_SAE.head()

,APPLY_DATE,VEH_SERV_ID,SERV_TRIP_SEQ,VEH_REGISTR_NUM,EVENT_DATETIME,START_DATETIME,END_DATETIME,START_VEH_LAT,START_VEH_LON,END_VEH_LAT,...,END_ROUTE_OFFSET_VALUE,TRIP_END_TYPE_CD,CREAT_USER_ID,CREAT_DATETIME,UPD_USER_ID,UPD_DATETIME,Servicio_orig,ServBus,HoraFinal,HoraFinal_segundos
0,20260519,AD0045123,1,10132,05:27:19,05:27:19,06:33:41,507235,588724,517432,...,23778.0,NaN,SmartCore,20260519052742,SmartCore,2.026052e+13,None,AD0045123,00:00:00,0
1,20260519,AD0045123,2,10132,06:35:57,06:35:57,07:34:26,517528,597469,507054,...,24797.0,NaN,SmartCore,20260519063600,SmartCore,2.026052e+13,None,AD0045123,00:00:00,0
2,20260519,AD0066049,1,7028,09:51:23,09:51:23,10:47:05,521962,605199,517528,...,22012.0,NaN,SmartCore,20260519095132,SmartCore,2.026052e+13,None,AD0066049,00:00:00,0
3,20260519,AD0066050,1,7028,19:45:59,19:45:59,20:42:49,520031,604881,517525,...,22012.0,NaN,SmartCore,20260519194600,SmartCore,2.026052e+13,None,AD0066050,00:00:00,0
4,20260519,AD0067054,1,40407,21:03:30,21:03:30,21:30:44,509019,600319,505311,...,24834.0,NaN,SmartCore,20260519210350,SmartCore,2.026052e+13,None,AD0067054,00:00:00,0


In [203]:
# Crear columna por defecto
eventos_SAE['Comparativa_Horas'] = np.nan

# Filtrar acciones == 5
acciones_5 = acciones_final[acciones_final['Id Acción'] == 5]

for _, fila in acciones_5.iterrows():
    serv = fila['Final']
    hora_ini = fila['HoraIniTeorica_segundos']
    
    # Filtrar en eventos_SAE el mismo servicio y horas mayores a hora_ini
    candidatos = eventos_SAE[
        (eventos_SAE['ServBus'] == serv) &
        (eventos_SAE['HoraFinal_segundos'] > hora_ini)
    ]
    
    if candidatos.empty:
        continue  # no hay nada para comparar
    
    # Tomar el primero posterior
    idx_min = candidatos['HoraFinal_segundos'].idxmin()
    hora_final = eventos_SAE.at[idx_min, 'HoraFinal_segundos']
    
    # Calcular diferencia
    diff = hora_final - hora_ini
    if diff > 1200:
        eventos_SAE.at[idx_min, 'Comparativa_Horas'] = 0
    else:
        eventos_SAE.at[idx_min, 'Comparativa_Horas'] = 1

eventos_SAE.head()


,APPLY_DATE,VEH_SERV_ID,SERV_TRIP_SEQ,VEH_REGISTR_NUM,EVENT_DATETIME,START_DATETIME,END_DATETIME,START_VEH_LAT,START_VEH_LON,END_VEH_LAT,...,TRIP_END_TYPE_CD,CREAT_USER_ID,CREAT_DATETIME,UPD_USER_ID,UPD_DATETIME,Servicio_orig,ServBus,HoraFinal,HoraFinal_segundos,Comparativa_Horas
0,20260519,AD0045123,1,10132,05:27:19,05:27:19,06:33:41,507235,588724,517432,...,NaN,SmartCore,20260519052742,SmartCore,2.026052e+13,None,AD0045123,00:00:00,0,NaN
1,20260519,AD0045123,2,10132,06:35:57,06:35:57,07:34:26,517528,597469,507054,...,NaN,SmartCore,20260519063600,SmartCore,2.026052e+13,None,AD0045123,00:00:00,0,NaN
2,20260519,AD0066049,1,7028,09:51:23,09:51:23,10:47:05,521962,605199,517528,...,NaN,SmartCore,20260519095132,SmartCore,2.026052e+13,None,AD0066049,00:00:00,0,NaN
3,20260519,AD0066050,1,7028,19:45:59,19:45:59,20:42:49,520031,604881,517525,...,NaN,SmartCore,20260519194600,SmartCore,2.026052e+13,None,AD0066050,00:00:00,0,NaN
4,20260519,AD0067054,1,40407,21:03:30,21:03:30,21:30:44,509019,600319,505311,...,NaN,SmartCore,20260519210350,SmartCore,2.026052e+13,None,AD0067054,00:00:00,0,NaN


In [204]:
# Crear columna por defecto
eventos_SAE['Comparativa_Horas'] = np.nan

# Filtrar acciones == 5
acciones_5 = acciones_final[acciones_final['Id Acción'] == 5]

for _, fila in acciones_5.iterrows():
    serv = fila['Final']
    hora_ini = fila['HoraIniTeorica_segundos']
    
    # Filtrar eventos del mismo servicio
    eventos_serv = eventos_SAE[eventos_SAE['ServBus'] == serv]
    
    # Filtrar solo los posteriores a hora_ini
    candidatos = eventos_serv[eventos_serv['HoraFinal_segundos'] > hora_ini]
    
    if candidatos.empty:
        # No hay más eventos → validar contra el último evento registrado
        if not eventos_serv.empty:
            hora_max = eventos_serv['HoraFinal_segundos'].max()
            if hora_ini > hora_max:
                # Caso especial → marcar 1 en el último registro
                idx_max = eventos_serv['HoraFinal_segundos'].idxmax()
                eventos_SAE.at[idx_max, 'Comparativa_Horas'] = 1
        continue
    
    # Tomar el primero posterior
    idx_min = candidatos['HoraFinal_segundos'].idxmin()
    hora_final = eventos_SAE.at[idx_min, 'HoraFinal_segundos']
    
    # Calcular diferencia
    diff = hora_final - hora_ini
    if diff > 1200:
        eventos_SAE.at[idx_min, 'Comparativa_Horas'] = 0
    else:
        eventos_SAE.at[idx_min, 'Comparativa_Horas'] = 1


eventos_SAE.head()

,APPLY_DATE,VEH_SERV_ID,SERV_TRIP_SEQ,VEH_REGISTR_NUM,EVENT_DATETIME,START_DATETIME,END_DATETIME,START_VEH_LAT,START_VEH_LON,END_VEH_LAT,...,TRIP_END_TYPE_CD,CREAT_USER_ID,CREAT_DATETIME,UPD_USER_ID,UPD_DATETIME,Servicio_orig,ServBus,HoraFinal,HoraFinal_segundos,Comparativa_Horas
0,20260519,AD0045123,1,10132,05:27:19,05:27:19,06:33:41,507235,588724,517432,...,NaN,SmartCore,20260519052742,SmartCore,2.026052e+13,None,AD0045123,00:00:00,0,NaN
1,20260519,AD0045123,2,10132,06:35:57,06:35:57,07:34:26,517528,597469,507054,...,NaN,SmartCore,20260519063600,SmartCore,2.026052e+13,None,AD0045123,00:00:00,0,NaN
2,20260519,AD0066049,1,7028,09:51:23,09:51:23,10:47:05,521962,605199,517528,...,NaN,SmartCore,20260519095132,SmartCore,2.026052e+13,None,AD0066049,00:00:00,0,NaN
3,20260519,AD0066050,1,7028,19:45:59,19:45:59,20:42:49,520031,604881,517525,...,NaN,SmartCore,20260519194600,SmartCore,2.026052e+13,None,AD0066050,00:00:00,0,NaN
4,20260519,AD0067054,1,40407,21:03:30,21:03:30,21:30:44,509019,600319,505311,...,NaN,SmartCore,20260519210350,SmartCore,2.026052e+13,None,AD0067054,00:00:00,0,NaN


In [205]:
acciones_final.to_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2025/Eliminaciones/{dia}_accionesregulacion_1.csv', index=False, sep=';')

In [206]:
eventos_SAE.to_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2025/Eliminaciones/{dia}_eventos_1.csv', index=False, sep=';')